In [25]:
import sys
sys.path.append("../src")

In [26]:
from datachecker.batch_sources import CsvBatchSource, BatchSource, JsonlBatchSource
from datachecker.schema_loader import YamlFileSchemaLoader, SchemaSpec, SchemaLoader
from datachecker.plans import PydanticPlanCompiler, PydanticPlan


In [27]:

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Protocol, Sequence

from pydantic import BaseModel, ValidationError

# ---- Common types ----
Record = dict[str, Any]  # (note: your sources yield Mapping[str, Any]; we normalize to dict for convenience)


# ---- Batch validation interfaces ----
class ValidationPlan(Protocol):
    ...


@dataclass(frozen=True)
class RowError:
    row_index: int                 # index within the *batch* (0..batch_size-1)
    record_hint: dict[str, Any]    # small subset of the record for debugging
    errors: list[dict[str, Any]]   # Pydantic-like error dicts


@dataclass(frozen=True)
class ValidationReport:
    schema_name: str
    total: int
    valid: int
    invalid: int
    row_errors: list[RowError]

    def ok(self) -> bool:
        return self.invalid == 0


class BatchValidator(Protocol):
    def validate_batch(self, plan: ValidationPlan, records: list[Record]) -> ValidationReport: ...



# ---- Concrete BatchValidator implementation ----
@dataclass(frozen=True)
class PydanticBatchValidator(BatchValidator):
    """
    Validates each record in the batch using plan.model.model_validate(...).

    - returns a ValidationReport (never raises due to validation errors)
    - collects per-row Pydantic errors
    """
    record_hint_keys: tuple[str, ...] = ("id", "email")  # what to include in report for quick debugging
    max_errors: int | None = None                        # stop collecting after N invalid rows (optional)

    def validate_batch(self, plan: PydanticPlan, records: list[Record]) -> ValidationReport:
        row_errors: list[RowError] = []
        valid = 0

        for i, rec in enumerate(records):
            # Ensure we pass a plain dict to pydantic (csv.DictReader returns dict[str, str|None] anyway)
            rec_dict = dict(rec)

            try:
                plan.model.model_validate(rec_dict)
                valid += 1
            except ValidationError as e:
                hint = {k: rec_dict.get(k) for k in self.record_hint_keys if k in rec_dict}
                row_errors.append(
                    RowError(
                        row_index=i,
                        record_hint=hint,
                        errors=e.errors(),  # list[dict], includes loc/msg/type/ctx
                    )
                )
                if self.max_errors is not None and len(row_errors) >= self.max_errors:
                    break

        total_checked = len(records) if self.max_errors is None else min(len(records), valid + len(row_errors))
        invalid = total_checked - valid

        return ValidationReport(
            schema_name=plan.name,
            total=total_checked,
            valid=valid,
            invalid=invalid,
            row_errors=row_errors,
        )


# ---- Example "engine" that validates whole files in batches ----
def validate_source_in_batches(
    source: Any,  # BatchSource (Protocol); kept Any so this snippet is standalone
    plan: PydanticPlan,
    validator: BatchValidator,
    batch_size: int = 1000,
) -> ValidationReport:
    """
    Validates all records from a source in batches and aggregates into one report.
    Assumes the source has read_batches(batch_size=...).
    """
    all_errors: list[RowError] = []
    total = valid = invalid = 0

    for batch_no, batch in enumerate(source.read_batches(batch_size=batch_size), start=1):
        report = validator.validate_batch(plan, [dict(r) for r in batch])

        total += report.total
        valid += report.valid
        invalid += report.invalid

        # shift row_index to be global (optional, but super useful)
        base = (batch_no - 1) * batch_size
        for re in report.row_errors:
            all_errors.append(
                RowError(
                    row_index=base + re.row_index,
                    record_hint=re.record_hint,
                    errors=re.errors,
                )
            )

    return ValidationReport(
        schema_name=plan.name,
        total=total,
        valid=valid,
        invalid=invalid,
        row_errors=all_errors,
    )

In [28]:

loader = YamlFileSchemaLoader(root_dir="schemas")
spec = loader.load("user")
plan = PydanticPlanCompiler().compile(spec)

src = JsonlBatchSource("data/users.jsonl")
validator = PydanticBatchValidator(record_hint_keys=("id", "email"), max_errors=50)

final_report = validate_source_in_batches(src, plan, validator, batch_size=500)

print("OK?", final_report.ok())
print("Total:", final_report.total, "Valid:", final_report.valid, "Invalid:", final_report.invalid)

# show first few errors
for err in final_report.row_errors[:3]:
    print("Row:", err.row_index, "hint:", err.record_hint)
    for e in err.errors:
        print("  ", e["loc"], "-", e["msg"])

OK? True
Total: 2000 Valid: 2000 Invalid: 0
